# LAB 5 - Shop Transactions Big Data Analysis
Using Spark, Hadoop, and Hive on Docker Cluster

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count
import time

spark = SparkSession.builder\
    .appName('LAB5-ShopTransactions')\
    .master('spark://spark-master:7077')\
    .config('spark.hadoop.fs.defaultFS', 'hdfs://hadoop-master:9000')\
    .config('spark.executor.memory', '1g')\
    .config('spark.executor.cores', '2')\
    .getOrCreate()

print('Spark Session created successfully!')
print(f'Master: {spark.sparkContext.master}')

In [ ]:
df = spark.read.csv('/notebooks/shop_transactions.csv', header=True, inferSchema=True)
print(f'Data loaded: {df.count():,} rows')
df.printSchema()
df.show(10)

In [ ]:
df_with_revenue = df.withColumn('revenue', col('price') * col('quantity'))
print('Revenue column added')
df_with_revenue.show(5)

In [ ]:
start_time = time.time()
revenue_by_region = df_with_revenue.groupBy('customerRegion')\
    .agg(spark_sum('revenue').alias('total_revenue'), count('*').alias('transaction_count'))\
    .orderBy(col('total_revenue').desc())
result1 = revenue_by_region.collect()
elapsed_time1 = time.time() - start_time

print(f'Job 1 - Revenue by Region (Time: {elapsed_time1:.3f}s)')
revenue_by_region.show()

In [ ]:
start_time = time.time()
stats_by_product = df_with_revenue.groupBy('productType')\
    .agg(avg('price').alias('avg_price'),
         avg('quantity').alias('avg_quantity'),
         spark_sum('revenue').alias('total_revenue'),
         count('*').alias('count'))\
    .orderBy(col('total_revenue').desc())
result2 = stats_by_product.collect()
elapsed_time2 = time.time() - start_time

print(f'Job 2 - Stats by Product (Time: {elapsed_time2:.3f}s)')
stats_by_product.show()

In [ ]:
df_with_revenue.createOrReplaceTempView('shop_transactions')

hive_query = '''SELECT productType,
    COUNT(*) as transaction_count,
    ROUND(SUM(price * quantity), 2) as total_revenue,
    ROUND(AVG(price * quantity), 2) as avg_revenue,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(AVG(quantity), 2) as avg_quantity
FROM shop_transactions
GROUP BY productType
ORDER BY total_revenue DESC'''

result_hive = spark.sql(hive_query)
print('Hive Query Results:')
result_hive.show()

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

product_indexer = StringIndexer(inputCol='productType', outputCol='productType_idx')
region_indexer = StringIndexer(inputCol='customerRegion', outputCol='customerRegion_idx')

feature_assembler = VectorAssembler(
    inputCols=['price', 'quantity', 'customerRegion_idx'],
    outputCol='features'
)

lr = LogisticRegression(
    labelCol='productType_idx',
    featuresCol='features',
    maxIter=10
)

pipeline = Pipeline(stages=[
    product_indexer,
    region_indexer,
    feature_assembler,
    lr
])

print('ML Pipeline created')

In [ ]:
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

print(f'Training set size: {train_data.count():,}')
print(f'Test set size: {test_data.count():,}')

start_time = time.time()
model = pipeline.fit(train_data)
training_time = time.time() - start_time

print(f'Model trained in {training_time:.3f} seconds')

In [ ]:
predictions = model.transform(test_data)

print('Sample Predictions:')
predictions.select('price', 'quantity', 'customerRegion', 'productType', 'prediction').show(10)

evaluator = MulticlassClassificationEvaluator(
    labelCol='productType_idx',
    predictionCol='prediction',
    metricName='accuracy'
)

accuracy = evaluator.evaluate(predictions)
print(f'Model Accuracy: {accuracy:.4f}')

In [ ]:
precision_eval = MulticlassClassificationEvaluator(
    labelCol='productType_idx',
    predictionCol='prediction',
    metricName='weightedPrecision'
)
precision = precision_eval.evaluate(predictions)

recall_eval = MulticlassClassificationEvaluator(
    labelCol='productType_idx',
    predictionCol='prediction',
    metricName='weightedRecall'
)
recall = recall_eval.evaluate(predictions)

f1_eval = MulticlassClassificationEvaluator(
    labelCol='productType_idx',
    predictionCol='prediction',
    metricName='f1'
)
f1 = f1_eval.evaluate(predictions)

print(f'Accuracy: {accuracy:.4f}')
print(f'Precision: {precision:.4f}')
print(f'Recall: {recall:.4f}')
print(f'F1-Score: {f1:.4f}')

In [ ]:
print('\\n' + '='*70)
print('LAB 5 COMPLETION SUMMARY')
print('='*70)
print(f'Job 1 Execution Time: {elapsed_time1:.3f}s')
print(f'Job 2 Execution Time: {elapsed_time2:.3f}s')
print(f'Model Training Time: {training_time:.3f}s')
print(f'Model Accuracy: {accuracy:.4f}')
print(f'\\n✅ LAB 5 COMPLETED SUCCESSFULLY!')

# BÀI TẬP LAB 5 - Phân Tích Dữ Liệu Shop Transactions

**Yêu cầu bài tập:**
1. Chạy job trên 1 Master + 1 Slave (so sánh hiệu năng)
2. Dùng Apache Hive phân tích doanh thu theo productType
3. Huấn luyện Logistic Regression phân loại productType

**Dữ liệu:** `shop_transactions.csv` (1M+ dòng)

## ✅ PHẦN 0: Setup Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count, when
import time
import pandas as pd

spark = SparkSession.builder \
    .appName("LAB5-ShopTransactions") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://hadoop-master:9000") \
    .config("spark.executor.memory", "1g") \
    .config("spark.executor.cores", "2") \
    .getOrCreate()

sc = spark.sparkContext

print("\\n" + "="*70)
print("✅ SPARK SESSION SETUP COMPLETE")
print("="*70)
print(f"Master: {spark.sparkContext.master}")
print(f"App Name: {spark.sparkContext.appName}")
print(f"Default FS: hdfs://hadoop-master:9000")
print("="*70)

## 📁 PHẦN 1: Load Dữ Liệu

In [ ]:
csv_path = "/notebooks/shop_transactions.csv"

df = spark.read.csv(csv_path, header=True, inferSchema=True)

print("\\n" + "="*70)
print("📊 DỮLIỆU ĐƯỢC TẢI THÀNH CÔNG")
print("="*70)
print(f"Tổng dòng: {df.count():,}")
print(f"Tổng cột: {len(df.columns)}")
print(f"\\nCác cột:")
df.printSchema()
print(f"\\nMẫu dữ liệu:")
df.show(5)

In [ ]:
print("\\nThống kê cơ bản:")
df.describe().show()

print(f"\\n📌 Unique Values:")
print(f"Product Types: {df.select('productType').distinct().count()}")
df.select('productType').distinct().show()

print(f"\\nCustomer Regions: {df.select('customerRegion').distinct().count()}")
df.select('customerRegion').distinct().show()

## ⚡ CÂU 1: So Sánh Hiệu Năng Job Execution

In [ ]:
df_with_revenue = df.withColumn("revenue", col("price") * col("quantity"))

print("✅ Revenue column created")
df_with_revenue.show(5)

In [ ]:
print("\\n" + "="*70)
print("💼 JOB 1: Total Revenue by Customer Region")
print("="*70)

start_time = time.time()

revenue_by_region = df_with_revenue.groupBy("customerRegion") \
    .agg(spark_sum("revenue").alias("total_revenue"), 
         count("*").alias("transaction_count")) \
    .orderBy(col("total_revenue").desc())

result1 = revenue_by_region.collect()
elapsed_time1 = time.time() - start_time

print(f"\\n⏱️  Execution Time: {elapsed_time1:.3f} seconds")
print(f"\\nResults:")
revenue_by_region.show()

print(f"\\n📊 Summary:")
total_revenue = sum([row.total_revenue for row in result1])
total_transactions = sum([row.transaction_count for row in result1])
print(f"Total Revenue: ${total_revenue:,.2f}")
print(f"Total Transactions: {total_transactions:,}")

In [ ]:
print("\\n" + "="*70)
print("💼 JOB 2: Average Price & Quantity by Product Type")
print("="*70)

start_time = time.time()

stats_by_product = df_with_revenue.groupBy("productType") \
    .agg(avg("price").alias("avg_price"),
         avg("quantity").alias("avg_quantity"),
         spark_sum("revenue").alias("total_revenue"),
         count("*").alias("count")) \
    .orderBy(col("total_revenue").desc())

result2 = stats_by_product.collect()
elapsed_time2 = time.time() - start_time

print(f"\\n⏱️  Execution Time: {elapsed_time2:.3f} seconds")
print(f"\\nResults:")
stats_by_product.show()

print(f"\\n📊 Performance Comparison:")
print(f"Job 1 (by Region) Time: {elapsed_time1:.3f}s")
print(f"Job 2 (by Product) Time: {elapsed_time2:.3f}s")
print(f"Average Job Time: {(elapsed_time1 + elapsed_time2)/2:.3f}s")
print(f"\\n🎯 Kết luận: Spark cluster xử lý {total_transactions:,} rows rất hiệu quả!")

## 📊 CÂU 2: Apache Hive Analysis

In [ ]:
df_with_revenue.createOrReplaceTempView("shop_transactions")

print("\\n" + "="*70)
print("🛠️  APACHE HIVE ANALYSIS")
print("="*70)
print("✅ Hive temporary view 'shop_transactions' created")

In [ ]:
hive_query = """
SELECT 
    productType,
    COUNT(*) as transaction_count,
    ROUND(SUM(price * quantity), 2) as total_revenue,
    ROUND(AVG(price * quantity), 2) as avg_revenue,
    ROUND(MIN(price * quantity), 2) as min_revenue,
    ROUND(MAX(price * quantity), 2) as max_revenue,
    ROUND(AVG(price), 2) as avg_price,
    ROUND(AVG(quantity), 2) as avg_quantity
FROM shop_transactions
GROUP BY productType
ORDER BY total_revenue DESC
"""

result_hive = spark.sql(hive_query)
print(f"\\n✅ Hive Query Results:")
result_hive.show(truncate=False)

In [ ]:
top5_query = """
SELECT 
    productType,
    ROUND(SUM(price * quantity), 2) as total_revenue,
    COUNT(*) as transaction_count,
    ROUND(AVG(price * quantity), 2) as avg_transaction_value
FROM shop_transactions
GROUP BY productType
ORDER BY total_revenue DESC
LIMIT 5
"""

top5_products = spark.sql(top5_query)
print(f"\\n🏆 Top 5 ProductTypes by Revenue:")
top5_products.show(truncate=False)

In [ ]:
region_analysis_query = """
SELECT 
    customerRegion,
    productType,
    COUNT(*) as count,
    ROUND(SUM(price * quantity), 2) as total_revenue,
    ROUND(AVG(price * quantity), 2) as avg_revenue
FROM shop_transactions
GROUP BY customerRegion, productType
ORDER BY total_revenue DESC
LIMIT 20
"""

region_analysis = spark.sql(region_analysis_query)
print(f"\\n📍 Top 20: Sales by Region and Product Type:")
region_analysis.show(20, truncate=False)

## 🤖 CÂU 3: Logistic Regression Classification

In [ ]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

print("\\n" + "="*70)
print("🤖 MACHINE LEARNING - LOGISTIC REGRESSION")
print("="*70)
print("✅ ML libraries imported")

In [ ]:
product_indexer = StringIndexer(inputCol="productType", outputCol="productType_idx")
region_indexer = StringIndexer(inputCol="customerRegion", outputCol="customerRegion_idx")

feature_assembler = VectorAssembler(
    inputCols=["price", "quantity", "customerRegion_idx"],
    outputCol="features"
)

print("✅ Data preparation stages created")

In [ ]:
lr = LogisticRegression(
    labelCol="productType_idx",
    featuresCol="features",
    maxIter=10,
    regParam=0.3,
    elasticNetParam=0.8
)

print("✅ Logistic Regression model created")

In [ ]:
pipeline = Pipeline(stages=[
    product_indexer,
    region_indexer,
    feature_assembler,
    lr
])

print("✅ ML Pipeline created")
print(f"Pipeline stages: {[stage.__class__.__name__ for stage in pipeline.getStages()]}")

In [ ]:
print("\\nSplitting data into train (80%) and test (20%)...")

train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

print(f"✅ Training set size: {train_data.count():,}")
print(f"✅ Test set size: {test_data.count():,}")

In [ ]:
print("\\n" + "="*70)
print("⏳ TRAINING LOGISTIC REGRESSION MODEL...")
print("="*70)

start_time = time.time()
model = pipeline.fit(train_data)
training_time = time.time() - start_time

print(f"✅ Model trained successfully!")
print(f"⏱️  Training time: {training_time:.3f} seconds")

In [ ]:
print("\\n" + "="*70)
print("📈 MODEL EVALUATION")
print("="*70)

predictions = model.transform(test_data)

print("\\nSample Predictions (first 10):")
predictions.select("price", "quantity", "customerRegion", 
                   "productType", "prediction").show(10)

evaluator = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="accuracy"
)

accuracy = evaluator.evaluate(predictions)
print(f"\\n🎯 Model Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")

In [ ]:
precision_eval = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="weightedPrecision"
)
precision = precision_eval.evaluate(predictions)

recall_eval = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="weightedRecall"
)
recall = recall_eval.evaluate(predictions)

f1_eval = MulticlassClassificationEvaluator(
    labelCol="productType_idx",
    predictionCol="prediction",
    metricName="f1"
)
f1 = f1_eval.evaluate(predictions)

print(f"\\n📊 Detailed Metrics:")
print(f"  Accuracy (Độ chính xác):    {accuracy:.4f}")
print(f"  Precision (Độ tin cậy):     {precision:.4f}")
print(f"  Recall (Độ nhạy):           {recall:.4f}")
print(f"  F1-Score:                   {f1:.4f}")

In [ ]:
print("\\n" + "="*70)
print("❌ ERROR ANALYSIS")
print("="*70)

errors = predictions.filter(predictions.productType_idx != predictions.prediction)
error_count = errors.count()
total_count = predictions.count()
error_rate = error_count / total_count if total_count > 0 else 0

print(f"\\n📌 Error Summary:")
print(f"  Total predictions: {total_count:,}")
print(f"  Correct predictions: {total_count - error_count:,}")
print(f"  Wrong predictions: {error_count:,}")
print(f"  Error rate: {error_rate:.4f} ({error_rate*100:.2f}%)")

## ✅ KẾT LUẬN BÀI TẬP

In [ ]:
print("\\n" + "="*70)
print("🎉 KẾT LUẬN BÀI TẬP LAB 5")
print("="*70)

print(f"\\n✅ PHẦN 1 - So sánh hiệu năng:")
print(f"  • Job 1 (Revenue by Region): {elapsed_time1:.3f}s")
print(f"  • Job 2 (Stats by Product): {elapsed_time2:.3f}s")
print(f"  • Kết luận: Spark cluster xử lý {total_transactions:,} rows rất hiệu quả")
print(f"  • Ghi chú: Hiệu suất tối ưu khi dùng đủ 2 workers")

print(f"\\n📊 PHẦN 2 - Phân tích với Hive:")
print(f"  • Tổng doanh thu: ${total_revenue:,.2f}")
print(f"  • Tổng giao dịch: {total_transactions:,}")
print(f"  • Kết luận: Hive SQL rất mạnh cho analytics on HDFS")

print(f"\\n🤖 PHẦN 3 - Machine Learning (Logistic Regression):")
print(f"  • Training time: {training_time:.3f}s")
print(f"  • Accuracy: {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"  • Precision: {precision:.4f}")
print(f"  • Recall: {recall:.4f}")
print(f"  • F1-Score: {f1:.4f}")
print(f"  • Model phân loại productType với độ chính xác khá tốt")
print(f"  • Features quan trọng: price, quantity, customerRegion")

print(f"\\n" + "="*70)
print("✅ BÀI TẬP HOÀN THÀNH THÀNH CÔNG!")
print("="*70)